In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/parvsxn/ai-layoffs/companies.csv
/kaggle/input/datasets/parvsxn/ai-layoffs/benefits.csv
/kaggle/input/datasets/parvsxn/ai-layoffs/industries.csv
/kaggle/input/datasets/parvsxn/ai-layoffs/tech_layoffs_hiring_trends_elite_v2.csv
/kaggle/input/datasets/parvsxn/ai-layoffs/company_industries.csv
/kaggle/input/datasets/parvsxn/ai-layoffs/job_skills.csv
/kaggle/input/datasets/parvsxn/ai-layoffs/job_industries.csv
/kaggle/input/datasets/parvsxn/ai-layoffs/company_specialities.csv
/kaggle/input/datasets/parvsxn/ai-layoffs/salaries.csv
/kaggle/input/datasets/parvsxn/ai-layoffs/skills.csv
/kaggle/input/datasets/parvsxn/ai-layoffs/postings.csv
/kaggle/input/datasets/parvsxn/ai-layoffs/employee_counts.csv


In [2]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [3]:
import pandas as pd


postings = pd.read_csv('/kaggle/input/datasets/parvsxn/ai-layoffs/postings.csv',
    usecols=['job_id','company_name','title','location','max_salary','min_salary',
              'normalized_salary','formatted_experience_level','listed_time',
              'remote_allowed','formatted_work_type'])

job_industries = pd.read_csv('/kaggle/input/datasets/parvsxn/ai-layoffs/job_industries.csv')
industries     = pd.read_csv('/kaggle/input/datasets/parvsxn/ai-layoffs/industries.csv')
job_skills     = pd.read_csv('/kaggle/input/datasets/parvsxn/ai-layoffs/job_skills.csv')
skills         = pd.read_csv('/kaggle/input/datasets/parvsxn/ai-layoffs/skills.csv')


job_industries = job_industries.merge(industries, on='industry_id', how='left')


job_skills = job_skills.merge(skills, on='skill_abr', how='left')


job_industries_agg = job_industries.groupby('job_id')['industry_name'] \
    .apply(lambda x: ', '.join(x.dropna().unique())).reset_index()

job_skills_agg = job_skills.groupby('job_id')['skill_name'] \
    .apply(lambda x: ', '.join(x.dropna().unique())).reset_index()


postings_full = postings.merge(job_industries_agg, on='job_id', how='left') \
                         .merge(job_skills_agg, on='job_id', how='left')

print(postings_full.shape)
postings_full.head()

(123849, 13)


,job_id,company_name,title,max_salary,location,min_salary,formatted_work_type,remote_allowed,formatted_experience_level,listed_time,normalized_salary,industry_name,skill_name
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,20.0,"Princeton, NJ",17.0,Full-time,NaN,NaN,1.713398e+12,38480.0,Real Estate,"Marketing, Sales"
1,1829192,NaN,Mental Health Therapist/Counselor,50.0,"Fort Collins, CO",30.0,Full-time,NaN,NaN,1.712858e+12,83200.0,NaN,Health Care Provider
2,10998357,The National Exemplar,Assitant Restaurant Manager,65000.0,"Cincinnati, OH",45000.0,Full-time,NaN,NaN,1.713278e+12,55000.0,Restaurants,"Management, Manufacturing"
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,175000.0,"New Hyde Park, NY",140000.0,Full-time,NaN,NaN,1.712896e+12,157500.0,Law Practice,Other
4,35982263,NaN,Service Technician,80000.0,"Burlington, IA",60000.0,Full-time,NaN,NaN,1.713452e+12,70000.0,Facilities Services,Information Technology


In [4]:
postings_full['listed_time'] = pd.to_datetime(postings_full['listed_time'], unit='ms', errors='coerce')


postings_full = postings_full[
    (postings_full['listed_time'] >= '2023-01-01') &
    (postings_full['listed_time'] <= '2026-06-01')
]

relevant_industries = ['Information Technology', 'Software Development', 'Computer Hardware',
                        'IT Services', 'Internet', 'Technology']  # adjust to actual values in your data
postings_full = postings_full[postings_full['industry_name'].isin(relevant_industries)]

print(postings_full.shape)  # should drop to a much more workable size

(3185, 13)


In [5]:
layoffs = pd.read_csv('/kaggle/input/datasets/parvsxn/ai-layoffs/tech_layoffs_hiring_trends_elite_v2.csv')
postings_full.to_csv('postings_clean.csv', index=False)
layoffs.to_csv('layoffs_clean.csv', index=False)

In [6]:
layoffs = pd.read_csv('/kaggle/input/datasets/parvsxn/ai-layoffs/tech_layoffs_hiring_trends_elite_v2.csv')

print(layoffs.shape)
layoffs.info()
layoffs.isnull().sum()

(12000, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   record_id               12000 non-null  object 
 1   company_name            12000 non-null  object 
 2   industry                12000 non-null  object 
 3   country                 12000 non-null  object 
 4   company_size            12000 non-null  object 
 5   month                   12000 non-null  object 
 6   year                    12000 non-null  int64  
 7   layoffs_count           12000 non-null  int64  
 8   layoff_percentage       12000 non-null  float64
 9   reason_for_layoffs      12000 non-null  object 
 10  ai_automation_impact    12000 non-null  float64
 11  ai_replacement_risk     12000 non-null  float64
 12  open_roles              12000 non-null  int64  
 13  hiring_trend            12000 non-null  object 
 14  remote_jobs_percentage  12

record_id                 0
company_name              0
industry                  0
country                   0
company_size              0
month                     0
year                      0
layoffs_count             0
layoff_percentage         0
reason_for_layoffs        0
ai_automation_impact      0
ai_replacement_risk       0
open_roles                0
hiring_trend              0
remote_jobs_percentage    0
top_hiring_role           0
stock_growth_percent      0
revenue_growth_percent    0
salary_budget_change      0
ai_adoption_level         0
employee_sentiment        0
job_security_score        0
market_condition          0
dtype: int64

In [7]:
layoffs['date'] = pd.to_datetime(
    layoffs['month'] + ' ' + layoffs['year'].astype(str),
    format='%b %Y',
    errors='coerce'
)

layoffs = layoffs.drop_duplicates()

layoffs['company_name'] = layoffs['company_name'].str.strip()
layoffs['industry'] = layoffs['industry'].str.strip().str.title()
layoffs['hiring_trend'] = layoffs['hiring_trend'].str.strip()

layoffs.head()

,record_id,company_name,industry,country,company_size,month,year,layoffs_count,layoff_percentage,reason_for_layoffs,ai_automation_impact,ai_replacement_risk,open_roles,hiring_trend,remote_jobs_percentage,top_hiring_role,stock_growth_percent,revenue_growth_percent,salary_budget_change,ai_adoption_level,employee_sentiment,job_security_score,market_condition,date
0,T0,Microsoft,Ai,Singapore,Enterprise,Mar,2026,860,1.8,AI Automation,6.4,5.0,5426,Moderate Hiring,46.7,ML Engineer,-25.7,30.3,4.9,4.4,8.7,8.6,Bull Market,2026-03-01
1,T1,Palantir,Ai,Canada,Big Tech,Feb,2024,955,1.8,Cost Cutting,0.9,1.1,9666,Moderate Hiring,58.9,ML Engineer,-5.6,6.1,1.5,1.0,8.2,7.2,Bull Market,2024-02-01
2,T2,Anthropic,Cybersecurity,USA,Mid-size,Apr,2025,18912,9.5,Overhiring Correction,7.1,3.9,437,Hiring Freeze,85.4,Frontend Developer,7.0,-23.6,-14.9,5.6,4.5,5.9,Recession,2025-04-01
3,T3,Spotify,Gaming,USA,Mid-size,Jun,2025,18159,9.1,Cost Cutting,10.4,7.4,1075,Hiring Freeze,44.0,Frontend Developer,31.6,-22.3,-1.6,6.5,5.4,4.7,Recession,2025-06-01
4,T4,Uber,Gaming,UK,Startup,Feb,2025,815,3.3,Market Slowdown,11.4,10.0,537,Moderate Hiring,53.2,Frontend Developer,85.3,26.6,9.8,9.3,6.7,5.8,Bull Market,2025-02-01


In [8]:
layoffs.to_csv('layoffs_clean.csv', index=False)

In [9]:
postings_full.to_csv('postings_clean.csv', index=False)
layoffs.to_csv('layoffs_clean.csv', index=False)

In [10]:
layoffs.shape
postings_full.shape
layoffs.columns.tolist()

['record_id',
 'company_name',
 'industry',
 'country',
 'company_size',
 'month',
 'year',
 'layoffs_count',
 'layoff_percentage',
 'reason_for_layoffs',
 'ai_automation_impact',
 'ai_replacement_risk',
 'open_roles',
 'hiring_trend',
 'remote_jobs_percentage',
 'top_hiring_role',
 'stock_growth_percent',
 'revenue_growth_percent',
 'salary_budget_change',
 'ai_adoption_level',
 'employee_sentiment',
 'job_security_score',
 'market_condition',
 'date']